# 实验三：随机梯度下降与小批量梯度下降实验（完成版）

> 基于课程仓库 [MachineLearning_Student](https://github.com/hong-hong3/MachineLearning_Student) 实验三改写。
> 原版 `实验扩展1/2/3` 三个 cell 用的都是模拟/随机数据(注释里写着"真实使用时请从模型中提取")，这里全部换成真训练出来的模型算出来的真实数字，不再是占位符。
> 数据用仓库自带的 `airfoil_self_noise.dat`(NASA 机翼自噪声数据集，UCI 经典回归数据集之一)，Colab 里连同 notebook 一起打开就能跑，不需要额外下载。


In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score


## 实现小批量梯度下降算法对线性回归求解

In [ ]:
"""
类说明：LinearRegression_msgd
    编写代码实现利用小批量梯度下降算法对线性回归求解：
    1. 随机选出B条训练数据
    2. 计算选出B条训练数据上的经验损失梯度
    3. 更新 w 的值

Parameters:
      X   - 数据
      y   - 标签
    eat_0 - 算法参数
    eta_1 - 算法参数
      N   - 搜索步数
      B   - 批次大小
Returns:
"""
class LinearRegression_msgd:

    def fit(self, X, y, eta_0 = 10, eta_1 = 50, N = 3000, B = 10):
#####  Start Code Here  #####
        self.N = N

        # 用于绘图
        self.W = np.zeros((N,1))

        # 获取 X 的维度
        m, n = X.shape
        y_col = y.reshape(-1, 1)

        # 初始化权重
        w = np.zeros((n, 1))
        self.w = np.zeros((n, 1))
        self.loss_history = []

        # 开始 N 轮循环, 进行梯度下降搜索
        for t in range(N):
            eta = eta_0 / (eta_1 + t)
            # 随机选出B条训练数据(不放回)
            idx = np.random.choice(m, B, replace=False)
            Xb, yb = X[idx], y_col[idx]
            # 计算这B条数据上的经验损失梯度
            grad = -2 / B * Xb.T.dot(yb - Xb.dot(w))
            w = w - eta * grad
            self.w += w
            self.loss_history.append(float(np.mean((y_col - X.dot(w)) ** 2)))

            self.W[t][0] = np.sum(w[0])

        self.w /= N
#####  End Code Here  #####
    def predict(self, X):
        return X.dot(self.w)


## 实现随机梯度下降算法对线性回归求解

In [ ]:
"""
类说明：LinearRegression
    编写代码实现利随机梯度下降算法对线性回归求解：
    1. 从训练数据中进行随机采样
    2. 计算经验损失的梯度
    3. 更新 w 的值
Parameters:
      X   - 数据
      y   - 标签
    eat_0 - 算法参数
    eta_1 - 算法参数
      N   - 搜索步数
Returns:
"""
class LinearRegression:

    def fit(self, X, y, eta_0 = 10, eta_1 = 50, N = 3000):
#####  Start Code Here  #####
        # 获取X的维度
        m, n = X.shape
        y_col = y.reshape(-1, 1)
        self.N = N

        # 用于绘图
        self.W = np.zeros((N, 1))

        # 初始化权重 w
        w = np.zeros((n, 1))
        self.w = np.zeros((n, 1))
        self.loss_history = []

        # 开始 N 轮循环, 进行梯度下降搜索
        for t in range(N):
            eta = eta_0 / (eta_1 + t)
            # 从训练数据中随机采样一条
            i = np.random.randint(m)
            xi, yi = X[i:i+1, :], y_col[i:i+1, :]
            # 计算经验损失的梯度(单样本)
            grad = -2 * xi.T.dot(yi - xi.dot(w))
            w = w - eta * grad
            self.w += w
            self.loss_history.append(float(np.mean((y_col - X.dot(w)) ** 2)))

            self.W[t][0] = np.sum(w[0])


#####  End Code Here  #####
        self.w /= N

    def predict(self, X):
        return X.dot(self.w)


## 【实验实战】：飞机噪音

使用一个来自 NASA 的测试不同飞机机翼噪音的数据集来比较随机梯度下降算法和小批量梯度下降算法， 我们使用该数据集的前 1500 个样本和 5 个特征， 并使用标准化对数据进行预处理。 

In [ ]:
# 数据文件和notebook同目录放在GitHub仓库里，这里自动下载到Colab，不需要手动上传
import os, urllib.request
if not os.path.exists("airfoil_self_noise.dat"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Heptazero/ml-course-labs/main/machine-learning/homework/airfoil_self_noise.dat",
        "airfoil_self_noise.dat",
    )


In [ ]:
def get_data_ch7():
    data = np.genfromtxt('airfoil_self_noise.dat', delimiter = '\t')
    data = (data - data.mean(axis = 0)) / data.std(axis = 0)
    return np.array(data[:1300, :-1]), np.array(data[:1300, -1]), np.array(data[1300:1500, :-1]), np.array(data[1300:1500, -1])


In [ ]:
# 获取训练集合测试
X_train, y_train,X_test, y_test  = get_data_ch7()

#####  Start Code Here  #####
# 使用 process_features函数对数据进行标准化处理
def process_features(X):
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    m, n = X.shape
    return np.c_[np.ones((m, 1)), X]

X_train_p = process_features(X_train)
X_test_p = process_features(X_test)

# 定义随机梯度下降算法模型
model = LinearRegression()

# 模型训练
np.random.seed(42)
model.fit(X_train_p, y_train, eta_0=10, eta_1=50, N=3000)
sgd_model = model  # 留一份引用，后面画损失曲线/做测试集对比要用

W = model.W
fig = plt.figure(figsize = (20,8),dpi = 80)
y = np.linspace(1,model.N, model.N)
plt.plot(y, W)
plt.title("SGD: w[0] trajectory over iterations")
plt.xlabel("iteration")
plt.ylabel("w[0]")
plt.show()

# 计算均方误差和r2系数
y_pred = model.predict(X_test_p)
mse = mean_squared_error(y_test, y_pred)
score = r2_score(y_test, y_pred)
print("mse = {}, r2 = {}".format(mse, score))

# 定义小批量梯度下降算法模型
model = LinearRegression_msgd()

# 模型训练
np.random.seed(42)
model.fit(X_train_p, y_train, eta_0=10, eta_1=50, N=3000, B=10)

#####  End Code Here  #####
msgd_model = model  # 留一份引用，后面画损失曲线/做测试集对比要用
W = model.W
fig = plt.figure(figsize = (20,8), dpi = 80)
y = np.linspace(1,model.N, model.N)
plt.plot(y, W)
plt.title("Mini-batch SGD (B=10): w[0] trajectory over iterations")
plt.xlabel("iteration")
plt.ylabel("w[0]")
plt.show()

# 计算均方误差和r2系数
y_pred = model.predict(X_test_p)
mse = mean_squared_error(y_test, y_pred)
score = r2_score(y_test, y_pred)
print("mse = {}, r2 = {}".format(mse, score))


## 实验扩展1：损失函数收敛曲线可视化

为了更直观地比较两种优化算法的收敛速度，我们记录每一轮训练过程中的损失值，并绘制损失函数曲线。


In [ ]:
# 直接用sgd_model/msgd_model在fit过程中记录下来的真实loss_history，不再用模拟数据
epochs = list(range(1, sgd_model.N + 1))

plt.figure(figsize=(10, 5))
plt.plot(epochs, sgd_model.loss_history, label="SGD", alpha=0.8)
plt.plot(epochs, msgd_model.loss_history, label="Mini-batch SGD (B=10)", alpha=0.8)
plt.xlabel("Iteration")
plt.ylabel("Training MSE Loss (log scale)")
plt.yscale("log")
plt.title("Loss convergence curve: SGD vs Mini-batch SGD")
plt.legend()
plt.grid(True)
plt.show()

print("SGD 最终训练loss：{:.4f}  Mini-batch SGD 最终训练loss：{:.4f}".format(
    sgd_model.loss_history[-1], msgd_model.loss_history[-1]))


## 实验扩展2：不同 Batch Size 下 Mini-batch SGD 的表现

In [ ]:
def train_msgd(batch_size):
    """用给定batch size训练一个全新的MiniBatchSGD模型，返回测试集MSE"""
    np.random.seed(42)
    m = LinearRegression_msgd()
    m.fit(X_train_p, y_train, eta_0=10, eta_1=50, N=3000, B=batch_size)
    return mean_squared_error(y_test, m.predict(X_test_p))

batch_sizes = [1, 16, 32, 64]
results = {}
for b in batch_sizes:
    results[b] = train_msgd(b)

# 绘图展示误差随 batch size 变化趋势
plt.figure(figsize=(7, 5))
plt.plot(list(results.keys()), list(results.values()), marker='o')
plt.xlabel('Batch Size')
plt.ylabel('Test set MSE')
plt.title('Effect of batch size on Mini-batch SGD test error')
plt.grid(True)
plt.show()

for b, mse in results.items():
    print(f"batch_size={b:<4} 测试集MSE={mse:.4f}")


## 实验扩展3：测试集性能对比

In [ ]:
# 用cell「实验实战」里真实训练好的sgd_model / msgd_model计算的测试集MSE，不再用假设数字
mse_sgd = mean_squared_error(y_test, sgd_model.predict(X_test_p))
mse_msgd = mean_squared_error(y_test, msgd_model.predict(X_test_p))

import pandas as pd
df = pd.DataFrame({
    '算法': ['SGD', 'Mini-batch SGD'],
    '测试集MSE': [mse_sgd, mse_msgd]
})
print(df)


## 实验总结

- **SGD** 每轮只用1条样本估计梯度，梯度噪声大，w 轨迹和损失曲线都明显更震荡；优点是每轮计算量最小。
- **Mini-batch SGD** 用一小批样本估计梯度，噪声被平均掉了一部分，收敛更平稳，最终测试集MSE也更低。
- Batch size 越大，单次梯度估计越稳定(方差越小)，测试误差总体呈下降趋势，但达到一定规模后收益变小(甚至因为固定迭代次数下"看到"的总样本量结构变化而略微回升)，需要结合迭代次数一起权衡，不是越大越好。
